# Customer Support Ticket Router with LangGraph

This notebook routes customer support tickets to billing or technical support using simple keyword rules. No LLM or API key is required. Run the code cells from top to bottom in Google Colab.

## 1. Install LangGraph
Install the library used to define the shared state, processing nodes, and routing edges. Run this setup cell when starting a fresh Colab runtime.

In [ ]:
# Run this setup cell first in Google Colab. No API key is needed.
%pip install -q langgraph


## 2. Define the State, Nodes, and Conditional Routing

`TicketState` stores the original `ticket`, its `category`, and the generated `reply`. Nodes return only the fields they change; LangGraph merges those updates into the state.

The `classify` node checks the lowercase ticket for billing keywords: `invoice`, `payment`, `refund`, `charged`, or `billing`. A matching substring selects billing; all other tickets go to technical support. This is a simple demonstration rule, so ambiguous tickets may be misclassified.

After classification, the conditional edge calls `route_ticket` to select either the `billing` node or the `technical` node. Each node supplies a fixed reply, then ends the workflow. `compile()` prepares the graph for execution.

**Flow:** Start → Classify → Billing or Technical → End.

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


# Shared state: each node reads the current values and returns updates.
class TicketState(TypedDict):
    ticket: str
    category: str
    reply: str


def classify(state: TicketState) -> dict:
    # Billing keywords take priority; other tickets go to technical support.
    billing_keywords = ('invoice', 'payment', 'refund', 'charged', 'billing')
    ticket = state['ticket'].lower()
    category = 'billing' if any(word in ticket for word in billing_keywords) else 'technical'
    return {'category': category}


# The conditional edge uses this function to choose the next node.
def route_ticket(state: TicketState) -> Literal['billing', 'technical']:
    return 'billing' if state['category'] == 'billing' else 'technical'


def billing(state: TicketState) -> dict:
    return {'reply': 'Our billing team can help. Please provide your invoice number.'}


def technical(state: TicketState) -> dict:
    return {'reply': 'Our technical team can help. Please share the error message and steps to reproduce it.'}


# Connect the nodes and compile the definition into an executable graph.
builder = StateGraph(TicketState)
builder.add_node('classify', classify)
builder.add_node('billing', billing)
builder.add_node('technical', technical)
builder.add_edge(START, 'classify')
builder.add_conditional_edges(
    'classify',
    route_ticket,
    {'billing': 'billing', 'technical': 'technical'},
)
builder.add_edge('billing', END)
builder.add_edge('technical', END)
graph = builder.compile()


## 3. Try One Ticket for Each Route

The first ticket contains `charged` and `invoice`, so it should reach billing. The second describes an app crash and has no billing keywords, so it should reach technical support.

Each `graph.invoke()` call starts with a new state and returns the completed state. The cell prints the original ticket, selected category, and support reply. Change the sample text to explore the keyword rules.

In [ ]:
# Run one sample ticket through each route.
sample_tickets = [
    'I was charged twice for my latest invoice.',
    'The app crashes whenever I try to sign in.',
]

for ticket in sample_tickets:
    result = graph.invoke({'ticket': ticket, 'category': '', 'reply': ''})
    print('Ticket:', result['ticket'])
    print('Category:', result['category'])
    print('Reply:', result['reply'])
    print()


## 4. Draw the Graph

Display the compiled graph as an image to see the classification step, both conditional routes, and the end of each path. The diagram shows the possible paths, rather than the execution trace of a particular ticket.

Image rendering uses an online Mermaid service and requires internet access. Only the graph structure is sent for rendering; ticket contents are not included.

In [ ]:
from IPython.display import Image, display

# Render the graph using the online Mermaid renderer; internet access is required.
# Only the graph structure is sent to the renderer, not the sample tickets.
display(Image(graph.get_graph().draw_mermaid_png()))
